# Activity-Based Weather Assessment

This notebook explores how forecast conditions are interpreted for different outdoor activities.

It defines activity profiles, evaluates temperature, wind speed, and precipitation probability, and produces structured assessments. The thresholds are application heuristics, not official safety standards.


## 1. Imports

We reuse the forecast DataFrame created in Notebook 01. If this notebook is opened independently, the fallback cell below creates a small sample forecast so that the activity logic can still be tested.

In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Activity profiles

Different activities emphasize different weather variables.

For example:

- **Cycling:** wind and rain are important.
- **Running:** temperature, rain and wind are considered.
- **Hiking:** temperature, precipitation and wind are considered.
- **Mountain biking:** precipitation and wind receive additional attention.
- **Camping:** overnight temperature, precipitation and wind are relevant.

These profiles implement the activity-specific idea described in initial design.

In [2]:
ACTIVITY_PROFILES = {
    "hiking": {
        "max_rain_probability": 60,
        "max_wind_kmh": 35,
        "low_temp_c": 0,
        "high_temp_c": 30
    },
    "cycling": {
        "max_rain_probability": 50,
        "max_wind_kmh": 30,
        "low_temp_c": 2,
        "high_temp_c": 32
    },
    "mountain biking": {
        "max_rain_probability": 45,
        "max_wind_kmh": 30,
        "low_temp_c": 0,
        "high_temp_c": 28
    },
    "running": {
        "max_rain_probability": 50,
        "max_wind_kmh": 30,
        "low_temp_c": 0,
        "high_temp_c": 30
    },
    "trail running": {
        "max_rain_probability": 45,
        "max_wind_kmh": 30,
        "low_temp_c": 0,
        "high_temp_c": 28
    },
    "camping": {
        "max_rain_probability": 50,
        "max_wind_kmh": 25,
        "low_temp_c": 5,
        "high_temp_c": 30
    },
    "outdoor tour": {
        "max_rain_probability": 50,
        "max_wind_kmh": 30,
        "low_temp_c": 2,
        "high_temp_c": 32
    }
}

ACTIVITY_PROFILES

{'hiking': {'max_rain_probability': 60,
  'max_wind_kmh': 35,
  'low_temp_c': 0,
  'high_temp_c': 30},
 'cycling': {'max_rain_probability': 50,
  'max_wind_kmh': 30,
  'low_temp_c': 2,
  'high_temp_c': 32},
 'mountain biking': {'max_rain_probability': 45,
  'max_wind_kmh': 30,
  'low_temp_c': 0,
  'high_temp_c': 28},
 'running': {'max_rain_probability': 50,
  'max_wind_kmh': 30,
  'low_temp_c': 0,
  'high_temp_c': 30},
 'trail running': {'max_rain_probability': 45,
  'max_wind_kmh': 30,
  'low_temp_c': 0,
  'high_temp_c': 28},
 'camping': {'max_rain_probability': 50,
  'max_wind_kmh': 25,
  'low_temp_c': 5,
  'high_temp_c': 30},
 'outdoor tour': {'max_rain_probability': 50,
  'max_wind_kmh': 30,
  'low_temp_c': 2,
  'high_temp_c': 32}}

## 3. Helper functions

These functions evaluate individual weather dimensions. Keeping the checks separate makes the logic easier to test and explain in the project documentation.

In [3]:
def assess_temperature(temperature, profile):
    if temperature < profile["low_temp_c"]:
        return "cold"
    if temperature > profile["high_temp_c"]:
        return "hot"
    return "comfortable"


def assess_wind(wind_speed, profile):
    if wind_speed > profile["max_wind_kmh"]:
        return "strong"
    if wind_speed > profile["max_wind_kmh"] * 0.7:
        return "moderate"
    return "light"


def assess_precipitation(precip_probability, profile):
    if precip_probability > profile["max_rain_probability"]:
        return "high"
    if precip_probability > profile["max_rain_probability"] * 0.5:
        return "moderate"
    return "low"


def assess_conditions(temperature, wind_speed, precip_probability, profile):
    return {
        "temperature": assess_temperature(temperature, profile),
        "wind": assess_wind(wind_speed, profile),
        "precipitation": assess_precipitation(
            precip_probability,
            profile
        )
    }

## 4. Activity assessment

The function below produces a structured result rather than a natural-language answer.

This is intentional: the **LLM will later turn this structured assessment into the final conversational response**.

In [4]:
def assess_activity_weather(
    activity,
    temperature,
    wind_speed,
    precipitation_probability,
    weather_description=None
):
    activity_key = activity.lower().strip()

    if activity_key not in ACTIVITY_PROFILES:
        raise ValueError(
            f"Unsupported activity: {activity}. "
            f"Available activities: {list(ACTIVITY_PROFILES.keys())}"
        )

    profile = ACTIVITY_PROFILES[activity_key]

    conditions = assess_conditions(
        temperature,
        wind_speed,
        precipitation_probability,
        profile
    )

    concerns = []

    if conditions["temperature"] == "cold":
        concerns.append("low temperature")

    if conditions["temperature"] == "hot":
        concerns.append("high temperature")

    if conditions["wind"] == "strong":
        concerns.append("strong wind")

    if conditions["precipitation"] == "high":
        concerns.append("high precipitation probability")

    if not concerns:
        summary = "No major weather concern identified by the prototype rules."

    else:
        summary = "Main factors: " + ", ".join(concerns) + "."

    return {
        "activity": activity_key,
        "temperature_c": temperature,
        "wind_kmh": wind_speed,
        "precipitation_probability": precipitation_probability,
        "weather_description": weather_description,
        "conditions": conditions,
        "concerns": concerns,
        "summary": summary
    }

## 5. Test with a single forecast record

If Notebook 01 has already been run in the same Colab session, this cell can use `forecast_df`.

Otherwise, the sample values provide a reproducible test.

In [5]:
# Use the first record from Notebook 01 when available.
if "forecast_df" in globals():
    test_row = forecast_df.iloc[0]

    test_temperature = float(test_row["temperature_2m"])
    test_wind = float(test_row["wind_speed_10m"])
    test_precip_probability = float(
        test_row["precipitation_probability"]
    )

    test_weather = str(
        test_row["weather_code"]
    )

else:
    # Fallback test values for independent execution.
    test_temperature = 18.0
    test_wind = 15.0
    test_precip_probability = 20.0
    test_weather = "2"

print("Temperature:", test_temperature)
print("Wind:", test_wind)
print("Precipitation probability:", test_precip_probability)

Temperature: 18.0
Wind: 15.0
Precipitation probability: 20.0


## 6. Evaluate the weather for hiking

In [6]:
hiking_result = assess_activity_weather(
    activity="hiking",
    temperature=test_temperature,
    wind_speed=test_wind,
    precipitation_probability=test_precip_probability,
    weather_description=test_weather
)

hiking_result

{'activity': 'hiking',
 'temperature_c': 18.0,
 'wind_kmh': 15.0,
 'precipitation_probability': 20.0,
 'weather_description': '2',
 'conditions': {'temperature': 'comfortable',
  'wind': 'light',
  'precipitation': 'low'},
 'concerns': [],
 'summary': 'No major weather concern identified by the prototype rules.'}

## 7. Compare several activities

The same forecast can produce different structured assessments because each activity has its own profile.

In [7]:
activities_to_test = [
    "hiking",
    "cycling",
    "mountain biking",
    "running",
    "camping",
    "trail running"
]

results = []

for activity in activities_to_test:
    result = assess_activity_weather(
        activity=activity,
        temperature=test_temperature,
        wind_speed=test_wind,
        precipitation_probability=test_precip_probability,
        weather_description=test_weather
    )

    results.append({
        "Activity": activity,
        "Temperature (°C)": test_temperature,
        "Wind (km/h)": test_wind,
        "Rain probability (%)": test_precip_probability,
        "Concerns": ", ".join(result["concerns"]) or "None"
    })

activity_results_df = pd.DataFrame(results)

activity_results_df

,Activity,Temperature (°C),Wind (km/h),Rain probability (%),Concerns
0,hiking,18.0,15.0,20.0,None
1,cycling,18.0,15.0,20.0,None
2,mountain biking,18.0,15.0,20.0,None
3,running,18.0,15.0,20.0,None
4,camping,18.0,15.0,20.0,None
5,trail running,18.0,15.0,20.0,None


## 8. Test an adverse-weather scenario

A controlled test helps demonstrate that the activity logic reacts to different forecast conditions.

In [8]:
adverse_test = assess_activity_weather(
    activity="cycling",
    temperature=8.0,
    wind_speed=38.0,
    precipitation_probability=75.0,
    weather_description="Heavy rain"
)

adverse_test

{'activity': 'cycling',
 'temperature_c': 8.0,
 'wind_kmh': 38.0,
 'precipitation_probability': 75.0,
 'weather_description': 'Heavy rain',
 'conditions': {'temperature': 'comfortable',
  'wind': 'strong',
  'precipitation': 'high'},
 'concerns': ['strong wind', 'high precipitation probability'],
 'summary': 'Main factors: strong wind, high precipitation probability.'}

## 9. Test a hiking scenario with moderate conditions

This example checks that the function can return a structured result without producing a false claim of guaranteed safety.

In [9]:
moderate_hiking_test = assess_activity_weather(
    activity="hiking",
    temperature=16.0,
    wind_speed=18.0,
    precipitation_probability=25.0,
    weather_description="Partly cloudy"
)

moderate_hiking_test

{'activity': 'hiking',
 'temperature_c': 16.0,
 'wind_kmh': 18.0,
 'precipitation_probability': 25.0,
 'weather_description': 'Partly cloudy',
 'conditions': {'temperature': 'comfortable',
  'wind': 'light',
  'precipitation': 'low'},
 'concerns': [],
 'summary': 'No major weather concern identified by the prototype rules.'}

## 10. Elevation context

An example request includes a scenario for hiking near Innsbruck at approximately **2,500 metres**.

The current prototype should treat the user's requested elevation as **additional context**. It must not pretend that the standard location forecast automatically represents the exact conditions at 2,500 m.

The application therefore store elevation explicitly and keep the limitation visible. A later version can use an elevation-aware weather request or a dedicated mountain-weather source if required.

In [10]:
def add_elevation_context(result, elevation_m=None):
    """
    Attach requested elevation as context.

    The current prototype does not independently calculate
    an elevation-adjusted temperature or wind forecast.
    """
    result = result.copy()
    result["requested_elevation_m"] = elevation_m

    if elevation_m is not None:
        result["elevation_note"] = (
            "Elevation was specified by the user. "
            "The current prototype does not independently "
            "calculate elevation-adjusted weather values."
        )
    else:
        result["elevation_note"] = None

    return result


innsbruck_context = add_elevation_context(
    moderate_hiking_test,
    elevation_m=2500
)

innsbruck_context

{'activity': 'hiking',
 'temperature_c': 16.0,
 'wind_kmh': 18.0,
 'precipitation_probability': 25.0,
 'weather_description': 'Partly cloudy',
 'conditions': {'temperature': 'comfortable',
  'wind': 'light',
  'precipitation': 'low'},
 'concerns': [],
 'summary': 'No major weather concern identified by the prototype rules.',
 'requested_elevation_m': 2500,
 'elevation_note': 'Elevation was specified by the user. The current prototype does not independently calculate elevation-adjusted weather values.'}

## 11. Activity assessment summary

The Activity Weather Logic now has a clear interface:

**Input**

- activity
- temperature
- wind speed
- precipitation probability
- weather description
- optional elevation context

**Output**

- structured activity assessment
- identified weather concerns
- explanation that can later be passed to the LLM

### Current architecture

```text
Weather API
    ↓
Structured forecast
    ↓
Activity Weather Logic
    ↓
Structured assessment
    ↓
LLM Weather Agent
    ↓
Natural-language response
```

### Next notebook

**Notebook 03 — LLM Weather Agent**

We will implement natural-language request understanding so a question such as:

> “Will it rain during my cycling tour in Berlin tomorrow afternoon?”

can be converted into structured parameters such as:

```python
{
    "activity": "cycling",
    "location": "Berlin",
    "date": "tomorrow",
    "time_period": "afternoon"
}
```

Then those parameters will be passed to the Weather Tool.